# Creating Dataset Splits

For AI/ML applications, datasets are typically split into training, validation,
and test sets. These splits are used to evaluate the generalization performance of
models and to prevent overfitting. However, the way these splits are created can
vary between datasets or when done on the fly, making it challenging to replicate
the behaviour across model training and evaluation runs.

For more consistent and reproducible results, the dataset should be split
deterministicly while accommodating for different split strategies.  

A counterintuitive example: the random split. Even for the random split, one could
create such a split randomly, then share *their* split with other so that
everyone has the same *random* split.

In this notebook, we will show you how to create these splits, archive them and share with the wider community.

## Lets start with loading the dataset

In [1]:
from proteingym.base import Dataset, Manifest
from proteingym.base.splits import RandomSplitter, KFoldSplitter

In [2]:
# lets load directly from manifest
manifest_path = "../example_data/neime_2019.toml"
manifest = Manifest.from_path(manifest_path)
dataset = Dataset.from_manifest(manifest)

## Split the dataset

We currently support two methods of splitting the dataset, RandomSplit and KFoldSplit.

In the RandomSplit we assign samples to each split randomly, in the KFold split we utilize [sklearns implementation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html) to generate folds.

In [3]:
# First we initialize the split with setttings
random_split_with_val = RandomSplitter(dataset, [0.8, 0.1, 0.1])
random_split_without_val = RandomSplitter(dataset, [0.8, 0.2])

In [4]:
# We can also initialize the KFoldSplit
kfold = KFoldSplitter(dataset, n_splits=5)

After initialization we can perform the split and create a superset of split datasets.

In [5]:
random_superset = random_split_with_val.split()

Similar for the kfolds:

In [6]:
kfold_superset = kfold.split()

We can view the the boolean masks of each slice:

In [7]:
kfold_superset.slices

[DatasetSlice(assays=[[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True,

And access specific slices:

In [8]:
kfold_superset.slices[0]

DatasetSlice(assays=[[True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, 

We can access the specific splits in two methods:

In [9]:
#method 1
train_ds, val_ds, test_ds = tuple(random_superset)

#method 2
for dataset in random_superset:
    #do something
    pass

To save the splits and share the splitted version of your dataset:

In [10]:
random_superset.dump()

PosixPath('/Users/vanderweg/Projects/PG2/proteingym-base/notebooks/NEIME_2019.splits.pgdata')

Now when we load the datasets the splits are already present in a splits.json config that gets packed automatically with the pgdata repository.

In [11]:
from proteingym.base.superset import Superset
my_split_dataset = Superset.from_path('NEIME_2019.splits.pgdata')

In [12]:
train_ds, val_ds, test_ds = tuple(my_split_dataset)

Combining this with the benchmark training entrypoints:

```python
def model(dataset: Superset):
    
    train_ds, val_ds, test_ds = tuple(dataset)

def train(dataset: Superset, model_card: ModelCard) -> 'MyModel':
    model = MyModel(model_card=model_card)

    model.train(dataset)
```